# Academic mobility and inbreeding-proxy analysis

This walkthrough ensures required generated upstream artifacts exist, then loads mobility inputs through the mobility stage API, computes the result in memory, and displays its result tables. If an upstream artifact is missing, its configured producer runs. The notebook does not write mobility outputs. The configured records are synthetic.

## What is being demonstrated?

Mobility links configured researcher-author selections to publication affiliations and institution metadata. It derives author-affiliation histories, publication mobility and coaffiliation measures, and an inbreeding proxy. This synthetic configuration disables university-affiliation detection, so the legacy first/last institution-overlap proxy is used. That proxy is not proof of education or training history.

In [1]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 100)

repo_root = Path.cwd()
while not (repo_root / "pyproject.toml").is_file() and repo_root != repo_root.parent:
    repo_root = repo_root.parent
if not (repo_root / "pyproject.toml").is_file():
    raise RuntimeError("Start Jupyter from within the academic-inbreeding repository.")
sys.path.insert(0, str(repo_root / "src"))

from acib.constants import AUTHOR_ID, RESEARCHER_ID, TITLE
from acib.mobility import compute_mobility, load_mobility_inputs
from acib.mobility.constants import (
    ACADEMIC_AGE_YEARS,
    AUTHOR_PROFILE_AMBIGUOUS,
    DATE_FIRST,
    DATE_LAST,
    FIRST_INST_SET,
    HAS_ANY_COAFFILIATION,
    HAS_PUBLICATION_MOBILITY,
    INBREEDING_DETECTION_METHOD,
    INBREEDING_DETECTION_STATUS,
    INBREEDING_MANUAL_REVIEW_REQUIRED,
    IS_INBREEDING_PROXY,
    LAST_INST_SET,
    N_AWAY_YEARS,
    N_COAFFILIATION_PUBLICATIONS,
    N_UNIQUE_AFFILIATIONS,
    PUBLICATION_HISTORY_YEARS,
    SHARED_INST_SET,
)
from acib.pipeline.config import load_mobility_config
from acib.pipeline.dependencies import ensure_stage_dependencies

## 1. Load mobility configuration

The loader resolves canonical upstream artifacts, external datasets, scientific options, and the output directory from the shared TOML. Required generated upstream artifacts are ensured before inputs load; the output directory is shown for transparency, and this notebook only computes and displays mobility results in memory.

In [2]:
config_path = repo_root / "config" / "pipeline_synthetic.toml"
mobility_config = load_mobility_config(config_path)
ensure_stage_dependencies("name_disambiguation", config_path)

print(f"Configuration: {config_path.relative_to(repo_root)}")
print(f"Configured mobility output directory (not written): {mobility_config.outputs.directory}")
print(f"University-affiliation rule: {mobility_config.enable_university_affiliation_inbreeding_detection}")
print(f"Professor merge: {mobility_config.enable_coaffiliated_professor_merge}")
print(f"Early-affiliation method: {mobility_config.early_affiliation_method}")
print(f"Minimum publication history: {mobility_config.min_publication_history_years}")

Configuration: config/pipeline_synthetic.toml
Configured mobility output directory (not written): /home/chri/PycharmProjects/academic-inbreeding/data/output/synthetic/mobility
University-affiliation rule: True
Professor merge: True
Early-affiliation method: first_n_years
Minimum publication history: 0


## 2. Load configured mobility inputs

Before loading inputs, the notebook ensures required generated upstream artifacts exist, running their configured producer stages if they are missing. The stage loader then reads and validates those selections and metadata, the external article-author evidence, and the affiliation GeoPackage. Missing external datasets remain errors. Existing artifacts are reused without freshness checks; no mobility analysis or output writing occurs in this step.

In [3]:
mobility_inputs = load_mobility_inputs(mobility_config)

print("Configured researcher-author selections:")
display(mobility_inputs.authors.head(10))
print("Configured article-author evidence:")
display(mobility_inputs.articles.head(10))
print("Configured geospatial affiliation crosswalk:")
display(mobility_inputs.affiliations.drop(
    columns=[mobility_inputs.affiliations.geometry.name], errors="ignore"
).head(10))
print(
    f"Loaded rows: selections={len(mobility_inputs.authors):,}; "
    f"articles={len(mobility_inputs.articles):,}; "
    f"affiliation features={len(mobility_inputs.affiliations):,}"
)

FileNotFoundError: Missing required mobility inputs:
- [name_disambiguation.outputs].final_author_selection (producer: name_disambiguation): /home/chri/PycharmProjects/academic-inbreeding/data/output/synthetic/name_disambiguation/final_author_selection.csv

## 3. Compute mobility from the configured inputs

`compute_mobility` applies configured metadata enrichment, optional professor-profile preparation, analysis settings, and output-contract validation. It returns the same in-memory result type used by the standalone workflow and does not persist mobility files. The preceding dependency check runs configured producers only for missing generated artifacts; missing external inputs remain errors, and existing artifacts are not checked for freshness.

In [ ]:
mobility_result = compute_mobility(mobility_inputs, mobility_config)

print(f"Professor profiles analyzed: {len(mobility_result.author_summary):,}")
print(f"Affiliation history rows: {len(mobility_result.author_affiliations):,}")
print(f"Per-profile article rows: {len(mobility_result.author_articles):,}")
print("Results remain in memory; no mobility files were written.")

## 4. Inbreeding proxy and affiliation evidence

The table below displays only profiles classified as positive inbreeding proxies. The complete 79-column result, including negative and unclassified profiles, remains available as `mobility_result.author_summary`.

In [ ]:
proxy = mobility_result.author_summary
positive_proxies = proxy.loc[proxy[IS_INBREEDING_PROXY].eq(True)]
requested_proxy_columns = [
    RESEARCHER_ID,
    AUTHOR_ID,
    TITLE,
    DATE_FIRST,
    DATE_LAST,
    FIRST_INST_SET,
    LAST_INST_SET,
    SHARED_INST_SET,
    IS_INBREEDING_PROXY,
    INBREEDING_DETECTION_METHOD,
    INBREEDING_DETECTION_STATUS,
    AUTHOR_PROFILE_AMBIGUOUS,
    PUBLICATION_HISTORY_YEARS,
    ACADEMIC_AGE_YEARS,
    HAS_PUBLICATION_MOBILITY,
    N_AWAY_YEARS,
    HAS_ANY_COAFFILIATION,
    N_COAFFILIATION_PUBLICATIONS,
    N_UNIQUE_AFFILIATIONS,
]
available_proxy_columns = [
    column for column in requested_proxy_columns
    if column in positive_proxies.columns
]
display(positive_proxies.loc[:, available_proxy_columns].sort_values(RESEARCHER_ID, kind="stable"))

## Summary

This notebook ensures missing generated upstream artifacts by running their configured producers, loads and validates mobility inputs through the mobility package API, calls `compute_mobility(inputs, config)`, and displays the returned in-memory summary, affiliation history, and per-profile article tables. It does not assemble the low-level analyzer arguments, manually restore upstream metadata, or write mobility outputs. Configured inputs use their canonical declarations; existing artifacts are reused without freshness detection.